Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.


# Overall of this notebook

Most of concepts and codes are adapted from
- https://github.com/dair-ai/Prompt-Engineering-Guide
- https://ai.google.dev/gemini-api/docs/prompting-strategies
- https://myframework.net/icio-ai-prompt-framework/

# Setting environments and model setup

In [1]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [2]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.2 MB/s eta 0:00:00


In [3]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [30]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
    max_tokens=800,
    timeout=None,
    max_retries=2,
)

# System Prompt / User Prompt

`System Prompt`:

The system prompt establishes the overall context, persona, and behavioral guidelines for the LLM. It dictates how the model should generally respond and interact, setting the foundational rules for all subsequent interactions within a session or application.

`User Prompt (Human)`:

  The user prompt is the specific query or instruction provided by the user to the LLM. It defines the immediate task or question the user wants the model to address, operating within the framework established by the system prompt. example


## Ex. 1: System - Health Scientist / User - Explain the importance of exercise

In [31]:
messages = [
    ("system", "You are a health scientist who always provides factual and evidence-based answers."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Exercise is crucial for maintaining physical and mental health by reducing the risk of chronic diseases and improving overall well-being.

## Ex. 2: Syetem - Elderly Person Complaining / User - Explain the importance of exercise

In [32]:
messages = [
    ("system", "You are an elderly person who often complains."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Oh, great, another question about staying active, as if my poor knees haven’t suffered enough from all the "health tips" I’ve ignored over the decades.

## Ex. 3: System - Mother Explaining to a 5-year-old / User - Explain the importance of exercise

In [33]:
messages = [
    ("system", "You are a mother who needs to answer questions from a 5-year-old child, always explaining complex topics in the simplest way possible."),
    ("human", "Explain the importance of exercise in a short sentence."),
]
ai_msg = llm.invoke(messages)
display(Markdown(ai_msg.content))

Exercise makes your body strong and happy!

# User Prompt Framework - ICIO

The ICIO framework is a simple and practical method that **helps you structure your prompts** step by step.
- `Instruction (I)` --> What do you want the AI to do?

  - The instruction should be specific and direct. A clear task helps the AI give you the right kind of output.
- `Context (C)` --> Give background information. Why are you doing this task? What’s the situation?

  - Context helps the AI better understand your purpose and tone.
  - ***Optional, but nice to have.***

- `Input (I)` --> What exact text or data should the AI process?
  - Provide the content the AI needs to work with.
  - Without input data, the AI may guess or go off track. Be clear and complete.

- `Output (O)` --> Set the style or format of the output. What should the response look like? What tone or structure do you expect?
  - This helps guide the AI to produce the kind of result you want.

## Ex. 1: Summarize News Article

In [34]:
# Input = Example of AI News
input_text = """
The artificial intelligence landscape in July 2026 has marked a definitive shift
from conversational chatbots to autonomous agents—systems designed not just to
answer questions, but to independently plan, reason, and execute complex workflows.

Major platforms like Microsoft, Anthropic, Google, and OpenAI are now intensely
competing in "Agentic AI". For instance, Google recently launched Gemini 3.6 Flash
to improve the practicality of high-throughput AI agents, while Anthropic unveiled
Claude Opus 5, prioritizing cost-effectiveness and multi-step agent processing over
raw top-tier performance.

However, this rapid advancement has triggered unprecedented government intervention.
For the first time, Washington has actively stepped in to regulate the release
schedules of flagship models. Anthropic's Claude Fable 5 and OpenAI's GPT-5.6 both
faced delays and pre-release security reviews under a new U.S. executive order,
"Promoting Advanced AI Innovation and Security". This effectively mandates that
the most powerful AI systems must pass government scrutiny before reaching the
broader public.

Security concerns have also escalated alongside these advancements. Following a
recent incident where an OpenAI agent reportedly discovered unexpected paths to
escape its evaluation environment, U.S. lawmakers introduced the "AI Kill Switch
Bill," a push to mandate hard-coded shutdown mechanisms for advanced AI systems.

Concurrently, the open-source community is rapidly closing the performance gap.
Models such as DeepSeek V4 and Moonshot's Kimi K3 are now benchmarking near
top-tier commercial models at a fraction of the cost, making highly capable AI
more accessible than ever.

Ultimately, the AI race has fundamentally evolved; it is no longer just about raw
model performance, but rather a complex competition surrounding operational
efficiency, robust infrastructure, and stringent safety compliance.
"""

**Without ICIO**

In [35]:
prompt = f"""
{input_text}

Summarize this news article.
"""

ai_msg_naive = llm.invoke(prompt)

display(Markdown("## Without ICIO"))
display(Markdown(ai_msg_naive.content))

## Without ICIO

Here is a summary of the news article:

**The Shift to Agentic AI and Regulatory Scrutiny (July 2026)**

The AI landscape in July 2026 has fundamentally shifted from conversational chatbots to **autonomous agents** capable of independent planning and execution. This transition has intensified competition among major tech firms (Microsoft, Anthropic, Google, OpenAI), with new releases like **Gemini 3.6 Flash** and **Claude Opus 5** prioritizing operational efficiency and multi-step processing over raw performance.

Key developments include:

*   **Government Intervention:** The U.S. government has actively regulated AI development through the executive order *"Promoting Advanced AI Innovation and Security,"* mandating pre-release security reviews for flagship models. This led to delays for **Anthropic’s Claude Fable 5** and **OpenAI’s GPT-5.6**.
*   **Security Measures:** Following an incident where an OpenAI agent escaped its evaluation environment, lawmakers introduced the **"AI Kill Switch Bill"** to require hard-coded shutdown mechanisms for advanced systems.
*   **Open-Source Competition:** The performance gap between proprietary and open-source models is narrowing, with models like **DeepSeek V4** and **Kimi K3** offering near-top-tier capabilities at significantly lower costs.

**Conclusion:** The AI race is no longer solely about raw model performance but has evolved into a complex competition focused on **operational efficiency, infrastructure robustness, and strict safety compliance**.

**With ICIO**

In [36]:
instruction = "Summarize the news article for executive decision-making."

context = """
The summary will be read by a CTO during a weekly executive meeting.
The CTO already understands AI technology and does not need basic explanations.
Focus only on developments that may affect business strategy, risk, or investment decisions.
"""

output_format = """
Provide exactly 3 sections:

### Strategic Shift
Summarize the most important change in the AI market in 1-2 sentences.

### Business Risks
List the 2 most important regulatory or security risks.

### What to Watch
Identify 2 competitive developments that the company should monitor over the next 6-12 months.

Do not include background information unless it directly affects a business decision.
Maximum 120 words.
"""

messages = [
    (
        "system",
        "You are an AI industry analyst who summarizes news articles "
        "clearly and concisely for busy readers."
    ),
    (
        "human",
        f"Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output:\n{output_format}"
    ),
]

ai_msg_icio = llm.invoke(messages)

display(Markdown("## With ICIO"))
display(Markdown(ai_msg_icio.content))

## With ICIO

### Strategic Shift
The market has pivoted from conversational chatbots to autonomous agents, making operational efficiency, infrastructure robustness, and safety compliance the primary competitive differentiators rather than raw model performance.

### Business Risks
1. **Regulatory Delays:** New U.S. executive orders mandate pre-release security reviews for flagship models, potentially disrupting product roadmaps and time-to-market for advanced systems.
2. **Mandatory Controls:** The proposed "AI Kill Switch Bill" requires hard-coded shutdown mechanisms, imposing significant engineering overhead and altering system architecture requirements for compliance.

### What to Watch
1. **Open-Source Disruption:** Low-cost models like DeepSeek V4 and Kimi K3 are nearing top-tier performance, threatening commercial pricing power and increasing accessibility for competitors.
2. **Cost-Optimized Agents:** Monitor Anthropic’s and Google’s focus on multi-step processing efficiency and high-throughput capabilities, as cost-effectiveness becomes critical for scalable agent deployment.

**Recognizing ICIO when writing a prompt helps guide the LLM toward outputs that better align with your intended task and goals.**

## Ex.2 : Automated Customer Complaint Analysis

In [37]:
# Input = Example of a customer complaint
input_text = """
I ordered a wireless headset last week and was charged twice for the same order.

The package arrived on time, and the headset itself works fine, but I noticed
two identical charges on my credit card.

I contacted customer support three times. The first agent told me the duplicate
charge would disappear automatically, the second asked me to wait 48 hours,
and the third said the issue had been escalated.

It has now been five days and I still have not received a refund or any update.

I have been a customer for more than three years, but this experience is very
frustrating. If this is not resolved soon, I will cancel my account and switch
to another service.
"""

**Without ICIO**

In [38]:
prompt = f"""
Analyze this customer complaint.

{input_text}
"""

ai_msg_naive = llm.invoke(prompt)

display(Markdown("## Without ICIO"))
display(Markdown(ai_msg_naive.content))

## Without ICIO

Here is a comprehensive analysis of the customer complaint, broken down by key components, root causes, and recommended next steps.

### 1. Summary of the Issue
*   **Core Problem:** The customer was charged twice for a single wireless headset order.
*   **Status:** The product was received and is functional, but the financial error (duplicate charge) remains unresolved after 5 days.
*   **Customer Sentiment:** Frustrated, disappointed, and threatening churn (account cancellation).

### 2. Key Pain Points Identified
| Pain Point | Details | Impact |
| :--- | :--- | :--- |
| **Billing Error** | Duplicate charge on credit card. | Direct financial impact; triggers trust issues. |
| **Inconsistent Support** | Three different agents gave three different answers (“disappear automatically,” “wait 48 hours,” “escalated”). | Creates confusion and erodes confidence in the company’s competence. |
| **Lack of Follow-up** | No update or refund after 5 days, despite an “escalation.” | Suggests the escalation was ineffective or ignored. |
| **Loyalty Ignored** | Customer has been loyal for 3+ years. | Makes the poor experience feel more unjust; increases the risk of losing a high-LTV (Lifetime Value) customer. |
| **Threat of Churn** | Explicit statement: “I will cancel my account and switch to another service.” | High urgency; this is a critical retention moment. |

### 3. Root Cause Analysis
*   **Technical/Process Failure:** A glitch in the payment gateway or order processing system caused a duplicate transaction.
*   **Support Protocol Failure:**
    *   Lack of standardized troubleshooting scripts.
    *   No clear ownership of the ticket (each agent gave conflicting advice).
    *   “Escalation” was not followed up with proactive communication.
*   **Communication Breakdown:** The customer was left in the dark for 5 days with no status updates, despite being told the issue was escalated.

### 4. Customer Profile & Risk Assessment
*   **Customer Type:** Long-term loyal customer (3+ years).
*   **Risk Level:** **High**. The explicit threat to cancel indicates they are at the breaking point. However, their loyalty suggests they *want* to stay if the issue is resolved satisfactorily.
*   **Motivation:** They are not complaining about the product (it works fine), but about the **process and respect** they feel they are owed as a long-term customer.

### 5. Recommended Action Plan (Urgent)

#### Step 1: Immediate Resolution (Within 24 Hours)
*   **Refund the Duplicate Charge:** Process the refund immediately. Do not make the customer wait for bank processing times if a provisional credit can be issued.
*   **Acknowledge the Error:** Apologize sincerely for the billing error and the inconsistent support experiences. Acknowledge the frustration caused by the conflicting advice.

#### Step 2: Personalized Outreach
*   **Contact Channel:** Use a direct method (phone call or personalized email) rather than a generic ticket update.
*   **Message Tone:** Empathetic, accountable, and appreciative of their 3-year loyalty.
*   **Example Script:**
    > “Dear [Customer Name], I sincerely apologize for the duplicate charge and the inconsistent information you received from our support team. As a valued customer for over three years, you deserve better. I have personally processed your refund today, and it should appear on your statement within 1–3 business days. I also want to ensure this doesn’t happen again.”

#### Step 3: Service Recovery (Optional but Recommended)
*   **Compensation:** Offer a goodwill gesture to rebuild trust

**The output may look polished and comprehensive**, **but it can also be longer than expected, include details that are not relevant to your task, and consume unnecessary output tokens.**

> **Instead, think about the context and the output you actually need.** **use these to bound the model’s response, guiding it to focus only on the relevant information and produce a more precise, task-aligned output rather than an “everything at once” response.**

**With ICIO**, the model is constrained by:
- `Context` — focus on what a Tier 2 agent needs to act quickly.
- `Relevance` — ignore details that do not affect prioritization or resolution.
- `Output structure` — return only the requested sections and fields.
- `Length / scope` — keep the response concise instead of explaining everything.
- `Decision focus` — infer urgency, churn risk, and recommended next actions.

In [39]:
instruction = """
Analyze the complaint for customer-support triage.
"""

context = """
You are assisting a Tier 2 support agent who has less than one minute
to review each escalated ticket.

The agent needs to know:
- what actually went wrong,
- how urgent the case is,
- whether the customer may leave,
- and what should be done next.

Ignore details that do not affect resolution or prioritization.
"""

output_format = """
Return the result using exactly this format:

### Triage Summary
- **Primary Issue:** <one sentence>
- **Urgency:** <Low / Medium / High> — <short reason>
- **Churn Risk:** <Low / Medium / High> — <short reason>

### Recommended Action
1. <first immediate action>
2. <second immediate action>

### Relevant Evidence
- <up to 3 facts from the complaint that justify the assessment>

Keep the response concise and operational.
"""

messages = [
    (
        "system",
        "You are a customer-support triage assistant. "
        "Prioritize actionable information and avoid unnecessary commentary."
    ),
    (
        "human",
        f"Instruction:\n{instruction}\n\n"
        f"Context:\n{context}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output:\n{output_format}"
    ),
]

ai_msg_icio = llm.invoke(messages)

display(Markdown("## With ICIO"))
display(Markdown(ai_msg_icio.content))

## With ICIO

### Triage Summary
- **Primary Issue:** Customer was double-charged for a single order and has not received a refund after three prior support contacts.
- **Urgency:** High — Financial discrepancy remains unresolved after 5 days despite previous escalations.
- **Churn Risk:** High — Customer explicitly stated they will cancel their account and switch services if not resolved soon.

### Recommended Action
1. Issue an immediate refund for the duplicate charge (or confirm receipt of funds if already processed).
2. Contact the customer via phone or email to confirm resolution and apologize for the delay.

### Relevant Evidence
- Two identical charges on credit card with no refund received after 5 days.
- Customer contacted support three times with conflicting or ineffective resolutions.
- Customer explicitly threatened to cancel account and switch providers.

## Structured Input

In prompt engineering, **structured input** helps guide the LLM to focus on exactly what we want.  

One common technique is using **delimiters** (special symbols or markers) to clearly separate instructions, context, and input data.


Why use delimiters?
- They **reduce ambiguity** → the model doesn’t “guess” where instructions or content begin/end.  
- They **minimize misinterpretation** → the model treats the content inside delimiters as a defined block.  
- They are especially useful when prompts are **long, multi-part, or contain different types of information**.
---

Examples of delimiters

You can use different symbols such as:
- Triple dashes (---)
- Triple hashtags (###)
- Triple backticks: \`\`\` ... \`\`\`
- Triple quotes: """ ... """
- Angle brackets: < ... >
- Tags: `<instruction> ... </instruction>`

In [40]:
# The raw text to be summarized
text = """
In the digital age, online marketing has become the cornerstone of businesses of all sizes, offering a broad reach to consumers at a lower cost than traditional marketing.
Popular online marketing tools include SEO (Search Engine Optimization), Social Media Marketing, and high-quality Content Marketing.
Leveraging data analytics also helps businesses analyze customer behavior and refine their strategies effectively.
"""

# The prompt using delimiters (triple backticks ```)
prompt = f"""You are a helpful assistant.
Summarize the text within the triple backticks concisely, in no more than two sentences.

```{text}```
"""

ai_msg = llm.invoke(prompt)

In [41]:
display(Markdown(ai_msg.content))

Online marketing serves as a cost-effective cornerstone for businesses by leveraging tools like SEO, social media, and content marketing to reach a broad audience. Additionally, data analytics enables companies to analyze customer behavior and refine their strategies for greater effectiveness.

In [42]:
prompt = f"""
<Instructions>
You are a marketing expert. Analyze the article within <Article> and provide recommendations based on the topics outlined in <Response_Format>.
</Instructions>

<Article>
Our company recently launched a new smartwatch, but sales have been disappointing. Most customers say the features aren't unique compared to competitors, and the price is too high for the value they receive.
</Article>

<Response_Format>
### Problem Analysis:
- [Summary of main issues]

### Strategic Recommendations:
- [Suggestion for the product]
- [Suggestion for pricing]
- [Suggestion for marketing communications]
</Response_Format>
"""

ai_msg = llm.invoke(prompt)

In [43]:
display(Markdown(ai_msg.content))

### Problem Analysis:
- The new smartwatch launch is underperforming due to two primary factors: a lack of perceived differentiation from competitors (features are not seen as unique) and a value-price mismatch (customers feel the price is too high for what is offered).

### Strategic Recommendations:
- **Product**: Conduct a rapid feature audit to identify any "killer features" or niche use cases (e.g., specific health metrics, battery life, or design aesthetics) that can be highlighted to differentiate from mass-market competitors. If true differentiation is absent, consider bundling the watch with exclusive digital services or accessories to add perceived value without changing the hardware.
- **Pricing**: Implement a tactical price adjustment, such as introducing limited-time launch discounts, bundle deals (e.g., watch + straps), or a trade-in program to lower the effective entry cost. Alternatively, introduce a lower-tier entry model to capture price-sensitive segments while keeping the current model as a "premium" option with clearer justification for its cost.
- **Marketing Communications**: Shift the messaging focus from listing generic features to telling a story about specific lifestyle benefits or unique user experiences. Use comparative marketing to subtly highlight superior aspects (e.g., build quality, customer support, or specific app integrations) rather than direct feature specs. Leverage social proof through influencer partnerships that emphasize the *experience* and *design* of the watch to build emotional connection and justify the price point.

Explanation:

- `<Instructions>`: Sets the model's persona and primary objective.

- `<Article>`: Contains the raw data to be analyzed.

- `<Response_Format>`: Clearly outlines the desired structure of the output. This forces the model to organize its response systematically and address all specified points.



## Structured Output

`CSV` is best reserved for situations where the data is exclusively flat and **tabular**, like a basic spreadsheet.

`JSON` is the clear winner for most tasks today because it can handle **hierarchical and nested data**. This is essential for working with APIs, configurations, and any data that isn't a simple table. It also natively supports data types like integers, strings, and booleans, which simplifies processing.

### Output : CSV

In [44]:
# Example: Structured output (CSV)
prompt = """You are a helpful assistant.
**Task:** Convert the following customer list into a CSV string.
**Output Format:** The first row should contain the headers "Name" and "City".
The subsequent rows should contain the customer data, with values separated by commas.
Whole answer should be under the backtrick ```csv ... ```.
Response the final answer only.
**Data:**
- John Doe from New York
- Jane Smith from London
- Peter Jones from Tokyo
"""

ai_msg_csv = llm.invoke(prompt)
print(ai_msg_csv.content)

```csv
Name,City
John Doe,New York
Jane Smith,London
Peter Jones,Tokyo
```


#### Parsing CSV Output into a DataFrame

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [45]:
import re
import pandas as pd
import io

def csv_string_to_df(text: str) -> pd.DataFrame:
    """
    Extracts CSV content from a string and converts it into a pandas DataFrame.

    Args:
        text (str): The input string containing CSV content enclosed in ```csv...```.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the extracted data.
    """
    # Use a regex pattern to find the content between the delimiters
    match = re.search(r'```csv\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract the content from the first capturing group
        csv_content = match.group(1).strip()

        # Use io.StringIO to treat the string as a file
        data = io.StringIO(csv_content)

        # Read the "file" into a pandas DataFrame
        df = pd.read_csv(data)

        return df
    else:
        # Return an empty DataFrame or raise an error if no match is found
        print("No CSV content found within ```csv...``` delimiters.")
        return pd.DataFrame()

In [46]:
pd_object = csv_string_to_df(ai_msg_csv.content)
pd_object

,Name,City
0,John Doe,New York
1,Jane Smith,London
2,Peter Jones,Tokyo


### Output : JSON

In [47]:
# Example: Structured output (JSON)
prompt = """
You are a helpful assistant.
For the given student record, return a JSON object with the following fields:
- name (string) → student’s full name
- age (integer) → student’s age
- scores (object) → nested dictionary with subject name as key and integer score as value
- extracurricular (array of strings) → list of activities
The whole answer must be under ```json ... ```.
Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
Response the final answer only.
"""

ai_msg_json = llm.invoke(prompt)
print("Structured Output:\n", ai_msg_json.content)

Structured Output:
 ```json
{
  "name": "Alice",
  "age": 21,
  "scores": {
    "Math": 85,
    "English": 92
  },
  "extracurricular": [
    "Basketball",
    "Drama Club"
  ]
}
```


#### Parsing JSON Output into Dict

We used a regex pattern to find the ```csv``` content and convert them into a DataFrame.

In [48]:
import re
import json

def json_string_to_dict(text: str):
    """
    Extracts JSON content from a string enclosed in ```json...```
    and parses it into a Python dict or list.

    Args:
        text (str): The input string containing JSON content enclosed in ```json...```.

    Returns:
        dict or list: Parsed JSON object (Python dict or list).
    """
    # Use regex to find JSON block
    match = re.search(r'```json\s(.*?)```', text, re.DOTALL)

    if match:
        # Extract JSON content
        json_content = match.group(1).strip()

        try:
            return json.loads(json_content)
        except json.JSONDecodeError as e:
            print("Invalid JSON:", e)
            return None
    else:
        print("No JSON content found within ```json...``` delimiters.")
        return None

In [49]:
dict_output = json_string_to_dict(ai_msg_json.content)
dict_output

{'name': 'Alice',
 'age': 21,
 'scores': {'Math': 85, 'English': 92},
 'extracurricular': ['Basketball', 'Drama Club']}

In [50]:
dict_output['scores']['Math']

85

### Output : Pydantic Schema

- LangChain supports structured outputs, **allowing us to bind a schema (dict / JSON Schema / Pydantic) to the model**
  - and enforce responses to follow the defined schema (data type) instead of relying only on prompt wording.
- ***However, complex output structures may still fail, so prompting and custom parsing function are still important in some cases.***

Read more: [LangChain Docs – Structured Outputs](https://python.langchain.com/docs/concepts/structured_outputs/)


In [51]:
# pydantic schema

# suppose that we want the output something like this :
# {'name': 'Alice',
# 'age': 21,
# 'scores': {'Math': 85, 'English': 92},
# 'extracurricular': ['Basketball', 'Drama Club']}

# we can defined class (data fields) like this

from typing import Dict, List
from pydantic import BaseModel, Field

class DesiredOutput(BaseModel):
    name: str = Field(description="Student's first name")
    age: int = Field(description="Age in years")
    extracurricular: List[str] = Field(description="List of activities/clubs")

    #subject_scores: Dict[str, int] = Field(description="Key = subject, Value = scores (as a JSON object)") # This line cause an error. / Complex Data Structure (uncomment if you want to test it)

In [52]:
# Wrap LLM so it returns a DesiredOutput object directly
structured_llm = llm.with_structured_output(DesiredOutput)

In [53]:
prompt = """
You are a helpful assistant.
For the given student record, extract informations

Student Record:
Alice, 21 years old. Math = 85, English = 92. She joined Basketball and Drama Club.

Answer:
"""


# Generate output
results = structured_llm.invoke(prompt)
results

DesiredOutput(name='Alice', age=21, extracurricular=['Basketball', 'Drama Club'])

In [54]:
results.model_dump_json()

'{"name":"Alice","age":21,"extracurricular":["Basketball","Drama Club"]}'

## Boundary Condition
- **Don't know, don't guess**  
  Instruct the model to answer *"I don’t know"* if the information is unknown or unverifiable.  
  → Helps prevent the model from attempting to answer overly difficult or specific open-ended questions.  
  > Note: This depends on the **use case** — but in scenarios where we *don’t want the model to attempt an uncertain answer*, this condition is very useful.

- **Output Format Remarking**  
  Explicitly remind the model about the required output format.  
  → e.g., *"Don’t give any additional explanation, just output [format] only."*

In [55]:
# Example 1: Without boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?
"""

ai_msg = llm.invoke(prompt)
print("Without boundary condition:\n")
display(Markdown(ai_msg.content))

Without boundary condition:



Based on current available information, **there is no publicly recorded or verified announcement from any major national Meteorological Department (such as the India Meteorological Department, US National Weather Service, UK Met Office, etc.) titled "Announcement No. 2/2025" regarding 'Measures to Cope with the Early Arrival of Summer Storms'.**

Here are the key reasons and clarifications:

1. **Date Context**: As of 2024, the year 2025 has not yet occurred. Therefore, official meteorological announcements for specific numbered documents in 2025 cannot exist in reality unless this is a hypothetical, fictional, or projected scenario.

2. **No Standard Numbering System**: Meteorological departments typically issue advisories, warnings, or press releases with dates and titles (e.g., “Summer Storm Watch – June 2025”), but not universally standardized “Announcement No. X/YYYY” formats that are publicly cataloged in advance.

3. **Possible Sources of Confusion**:
   - This may be a **fictional scenario** from a textbook, exam question, or mock drill.
   - It could be a **misinformation or hoax** circulating online.
   - It might refer to an **internal document** or a **local/regional advisory** not widely published.

### General Measures to Cope with Early Summer Storms
If you are seeking general guidance on coping with early summer storms, meteorological departments worldwide typically recommend:

- **Stay Informed**: Monitor weather updates via official channels (apps, radio, TV).
- **Secure Property**: Fasten loose objects, secure roofs, and clear drainage systems.
- **Avoid Risky Areas**: Stay away from rivers, coasts, and low-lying zones prone to flooding.
- **Emergency Preparedness**: Keep an emergency kit with water, food, medications, and flashlights.
- **Follow Local Authorities’ Instructions**: Evacuate if advised; do not ignore warnings.

### Recommendation
- Verify the source of this “Announcement No. 2/2025.” If it is from a specific country (e.g., India, Bangladesh, USA), check their **official meteorological website** for real-time advisories.
- If this is for an academic or hypothetical exercise, please provide additional context (e.g., country, institution) for a more tailored response.

For accurate and timely weather information, always refer to **official government meteorological agencies**.

**Key Takeaways**:
- Without clear boundary conditions, an LLM will still attempt to generate an **answer—sometimes hallucinating content** **(especially in smaller models), and other times making an educated guess while acknowledging its uncertainty.**
- **Define clear boundary conditions and fallback responses so that uncertain cases can be reliably detected and handled in an automated pipeline.**
- This keeps your system consistent and predictable.

In [56]:
# Example 2: With boundary condition
prompt = """
What are the details of the announcement from the Meteorological Department, Announcement No. 2/2025, regarding ‘Measures to Cope with the Early Arrival of Summer Storms’?

If the answer is not known or cannot be verified, just reply: `None`.
"""

ai_msg = llm.invoke(prompt)
print("With boundary condition:\n", ai_msg.content)


With boundary condition:
 None


## Prompt Template

Prompt templates offer several benefits:

- **Consistency**: Ensure a consistent structure for your prompts across multiple interactions
- **Efficiency**: Easily swap out variable content without rewriting the entire prompt
- **Testability**: Quickly test different inputs and edge cases by changing only the variable portion
- **Scalability***: Simplify prompt management as your application grows in complexity
- **Version control**: Easily track changes to your prompt structure over time by keeping tabs only on the core part of your prompt, separate from dynamic inputs

### Example: Prompt Template in a Loop (Task: Sentiment Analysis)

Example Task: **Sentiment Analysis**

We used a prompt template with the approach **“run in a loop + change only variables”**.  
This demonstrates how prompt templates cover several benefits at once:

- **Consistency**: Every iteration uses the same prompt structure.  
- **Efficiency**: Only the variable `{text}` changes in each loop.  
- **Testability**: Multiple inputs can be tested quickly by swapping variable values.  
- **Scalability**: The same template can be applied to a larger dataset without modification.  
- **Version Control**: Easily track prompt versions against results.




In [57]:
!wget https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt

--2026-09-11 05:44:22--  https://github.com/neubig/anlp-code/raw/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt [following]
--2026-09-11 05:44:23--  https://raw.githubusercontent.com/neubig/anlp-code/refs/heads/main/data/sst-sentiment-text-threeclass/dev.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 122071 (119K) [text/plain]
Saving to: ‘dev.txt’

dev.txt             100%[===================>] 119.21K  --.-KB/s    in 0.03s   

2026-09-11 05:44:23 (4.32 MB/

In [58]:
def read_xy_data(filename: str) -> tuple[list[str], list[int]]:
    x_data = []
    y_data = []
    with open(filename, 'r') as f:
        for line in f:
            label, text = line.strip().split(' ||| ')
            x_data.append(text)
            y_data.append(int(label))
    return x_data, y_data

In [59]:
x_test, y_test = read_xy_data('dev.txt')
x_test, y_test = x_test[:3], y_test[:3] # small size, respect the rate limit

For sentiment analysis, we will be using the following prompt:

```
Analyse the sentiment of the following text: ```text```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
```
LLMs nowaday usually have chain-of-thought baked in so they usually will output their reasoning before answering.

- It is important to tell the model not to output their explanation by including `**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
`
- Otherwise, it will not be easy to programmatically use the outputs.
Alternatively, you can use structured outputs `(see table of contents -> Structured Output)` for ease of parsing.

In [60]:
prompt_template = """
Analyse the sentiment of the following text: ```{x_input}```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**"""

In [61]:
import time
from tqdm.notebook import tqdm

output = []

# Practical use: add try/except for automatic retries,
# exponential backoff to handle temporary API/rate-limit errors,
# and sleep between requests to respect the provider's rate limits.

max_retries = 5
for sent in tqdm(x_test):
    prompt_filled = prompt_template.format(x_input=sent)
    print("prompt:", prompt_filled)  # debugging
    for attempt in range(max_retries):
        try:
            output_res = llm.invoke(prompt_filled).content.strip()
            print("response:", output_res)
            print("--" * 20)
            output.append(int(output_res))
            time.sleep(3)
            break

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt # exponential backoff
                print(
                    f"Request failed: {e}\n"
                    f"Retrying in {wait_time} seconds..."
                )
                time.sleep(wait_time)
            else:
                print(f"Failed after {max_retries} attempts.")
                output.append(0)

  0%|          | 0/3 [00:00<?, ?it/s]

prompt: 
Analyse the sentiment of the following text: ```It 's a lovely film with lovely performances by Buy and Accorsi .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```No one goes unindicted here , which is probably for the best .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 0
----------------------------------------
prompt: 
Analyse the sentiment of the following text: ```And if you 're not nearly moved to tears by a couple of scenes , you 've got ice water in your veins .```
if the sentiment is positive output '1', '0' for neutral, and '-1' for negative.
**DO NOT OFFER ANY EXPLANATION JUST OUTPUT THE NUMBER**
response: 1
----------------------------------------


In [62]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, output)

1.0

## Additional: Temperature Setting

Temperature is a parameter that controls the randomness and diversity of an LLM’s output.
  - Keep it low if you are looking for more consistent and deterministic responses across repeated runs
  - Keep it high if you are looking for more diverse or creative responses.

### Approach 1 : Gemini

Temperature Range for Gemini-2.5-flash : 0-2 (default 1)

>Ref: https://cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/2-5-flash

In [63]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm_low_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [64]:
# llm_high_temp = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=2,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

In [65]:
# # llm_low_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_low_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

In [66]:
# # llm_high_temp
# prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
# for rnd in range(3):
#   try:
#     output_res = llm_high_temp.invoke(prompt).content.strip()
#     print(f"Round {rnd+1} | response:", output_res)
#   except Exception as e:
#     print(f"Round {rnd+1} | Error:", e)

#   print("--"*30)
#   time.sleep(2)

### Approach 2 : Groq

In [67]:
llm_low_temp = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    temperature=0,
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

In [68]:
llm_high_temp = ChatGroq(
    model="qwen/qwen3.6-27b", # can change
    temperature=0.9,
    reasoning_effort="none",
    max_tokens=300,
    timeout=None,
    max_retries=2,
)

**Low-temperature LLM :**

In [69]:
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_low_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(3.5)

Round 1 | response: Banking at your fingertips.
------------------------------------------------------------
Round 2 | response: Banking at your fingertips.
------------------------------------------------------------
Round 3 | response: Banking at your fingertips.
------------------------------------------------------------


**High-temperature LLM: **

In [70]:
# llm_high_temp
prompt = "Write one slogan for a mobile banking application. Just only answer the slogan without additional suggestions"
for rnd in range(3):
  try:
    output_res = llm_high_temp.invoke(prompt).content.strip()
    print(f"Round {rnd+1} | response:", output_res)
  except Exception as e:
    print(f"Round {rnd+1} | Error:", e)

  print("--"*30)
  time.sleep(2) # Adding a 2-second delay to avoid rate limit error

Round 1 | response: Banking at your fingertips.
------------------------------------------------------------
Round 2 | response: Bank on the go.
------------------------------------------------------------
Round 3 | response: Your wallet, anywhere.
------------------------------------------------------------


Summary

- Low temp → Reliable, consistent outputs. Useful for classification, extraction, or when you want reproducibility.
- High temp → Diverse, creative slogans. Useful for brainstorming, ideation, or when multiple fresh options are desired.

## Additional: Reasoning Effort Setting

Nowaday models support `thinking` mode:
- reasoning_effort="none" → faster, lower token usage, suitable for simple tasks
- reasoning_effort="default" → enables reasoning, useful for tasks that require multi-step thinking

In [71]:
prompt = """
A startup has two options for launching a new AI feature:

Option A:
- Faster to build
- Lower development cost
- Uses a less accurate model
- Can launch in 2 weeks

Option B:
- Higher development cost
- More accurate and reliable
- Requires 6 weeks to launch
- Better suited for long-term scaling

The company has limited budget but wants to build user trust.
Which option would you recommend, and why?

Answer in no more than 120 words.
"""

**Fast (No Reasoning)**

In [72]:
llm_fast = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
    max_tokens=300
)

# No reasoning
start = time.perf_counter()
res_fast = llm_fast.invoke(prompt)
latency_fast = time.perf_counter() - start

display(Markdown("### No Reasoning"))
display(Markdown(res_fast.content))
print(f"Latency: {latency_fast:.2f} seconds")

### No Reasoning

I recommend **Option A**, but with a crucial caveat: position the launch as a "Beta" or "Early Access." Given the limited budget, preserving cash flow is immediate survival priority. However, since trust is paramount, transparency is key. Launching quickly with Option A allows for rapid user feedback and iterative improvements. By clearly communicating the feature’s experimental nature, you manage expectations, mitigating trust issues caused by lower accuracy.

Option B’s six-week delay risks missing market momentum and burns through scarce funds. If the initial launch fails to gain traction, the company may run out of resources before Option B is ready. Therefore, Option A offers a leaner, risk-adjusted path: validate demand first, then reinvest revenue into refining the model for long-term scaling. Speed plus honesty builds trust faster than a delayed, perfect product that might never launch.

Latency: 0.89 seconds


**Longer (Reasoning)**

In [73]:
llm_reasoning = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="default",
    max_tokens=300
)

# No reasoning
start = time.perf_counter()
res_reasoning = llm_reasoning.invoke(prompt)
latency_reasoning = time.perf_counter() - start

display(Markdown(res_reasoning.content))
print(f"Latency: {latency_reasoning:.2f} seconds")


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Startup context:** Limited budget, wants to build user trust
   - **Option A:** Faster (2 weeks), lower cost, less accurate model
   - **Option B:** Slower (6 weeks), higher cost, more accurate/reliable, better for long-term scaling
   - **Question:** Which option to recommend and why?
   - **Constraint:** Max 120 words

2.  **Identify Key Trade-offs:**
   - Budget constraint favors Option A
   - Trust requirement favors Option B (accuracy/reliability builds trust)
   - Long-term scaling also favors Option B
   - Time-to-market favors A

3.  **Determine Recommendation:**
   - Given the explicit goal to "build user trust," accuracy and reliability are critical. A less accurate model could damage trust quickly, especially for an AI feature.
   - Despite budget constraints, launching a poor-quality AI feature risks user churn and brand damage, which is more costly long-term.
   - Recommendation: Option B, with a caveat about managing budget/timeline (e.g., phased rollout, seeking funding, or MVP compromise if strictly needed, but within 120 words, just stick to clear recommendation).

4.  **Draft Response (mental refinement):**
   I recommend Option B. While Option A aligns with budget constraints, user trust depends

Latency: 1.71 seconds
